# PS3 — Root-Cause Taxonomy & Gap Analysis (all three fleets)

Built from the persisted spine. **No Spark needed** — reads the parquet directly.

## Why this is not three trained classifiers

`component_subsystem` is a property of the **event type**: `silver.dim_event_type`
(S07) supplies it, joined on `EVENT_TYPE_ID`. A classifier handed
`event_type_id` / `event_type_name` / `component_type_name` reconstructs the
target exactly and scores ~100% meaninglessly — the same defect this programme
already documents in PS4 (*"spark_gbt reports test AP 0.9994 … that gap IS the
tell"*). Strip those columns and you are predicting which component failed from
hour-of-day and station, which is genuinely weak.

So root-cause **attribution** is a lookup, not a prediction. The device states
which subsystem failed. **The product value is coverage**, and that is precisely
what demonstrates the gap.

The **predictive** product — *"PS1 says this device fails in 3 days; which
component will it be?"* — is a separate build with the current event's type
excluded. It chains onto PS1 and has no leakage. Worth doing, not today.

## What this produces

1. **Taxonomy** — every class, every fleet, side by side
2. **Diagnostic richness** — entropy and *effective classes*, because a count of
   classes overstates how much a dominated taxonomy actually tells you
3. **The gap** — ServiceNow coverage vs device coverage per fleet
4. **Agreement** — where ServiceNow did answer, does the device concur
5. **CSV + JSON export** for the deck and the dashboard

## Expected headline

| fleet | classes | effective | dominant |
|---|---|---|---|
| TVM | 7 | **3.56** | CSC_READER 49.6% |
| GATE | 4 | **1.58** | DOPP 86.6% |
| VALIDATOR | 3 | **1.64** | COMMS 80.7% |

Effective classes = 2^entropy — the number of *equally likely* classes carrying
the same information. TVM supports real triage. GATE and VALIDATOR name a
component but barely discriminate, and saying so first is stronger than being
asked.


In [ ]:
# =============================================================================
# PS3 -- COMPARATIVE TAXONOMY + GAP ANALYSIS  (all three fleets)
#
# WHAT THIS IS, AND WHAT IT IS NOT.
#
# It is NOT a trained model, and deliberately so. component_subsystem is a
# property of the EVENT TYPE -- silver.dim_event_type (S07) supplies it, joined
# on EVENT_TYPE_ID. A classifier given event_type_id / event_type_name /
# component_type_name would reconstruct the target exactly and score ~100%
# meaninglessly. That is the defect this programme already documents in PS4
# ("spark_gbt reports test AP 0.9994 ... that gap IS the tell").
#
# So root-cause ATTRIBUTION is a lookup, not a prediction: the device states
# which subsystem failed. The product value is COVERAGE -- 97% of OOS device-days
# versus ServiceNow's 1.9% on gates and 0% on validators -- and that is exactly
# what a client needs to see.
#
# The PREDICTIVE product ("PS1 says this fails in 3 days; which component?") is a
# separate build with the current event's type excluded. Different notebook.
#
# Reads the persisted spine parquet. No Spark required.
# =============================================================================
import pandas as pd, numpy as np, os, json

OUT = "PS3_spine_outputs"
SUB = {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"}
frames_pd = {f: pd.read_parquet(f"{OUT}/{s}_ps3_spine.parquet") for f, s in SUB.items()}
for f, d in frames_pd.items():
    print(f"  {f:10s} {len(d):>9,} rows, {len(d.columns)} cols")


In [ ]:
# -----------------------------------------------------------------------------
# 1. THE TAXONOMY -- every class, every fleet, side by side
# -----------------------------------------------------------------------------
print("\n" + "=" * 84)
print("PS3 ROOT-CAUSE TAXONOMY -- device-native, all fleets")
print("=" * 84)

tax = {}
for fleet, d in frames_pd.items():
    lab = d[d["root_cause_state"] == "LABELLED"]
    vc = lab["root_cause"].value_counts()
    tax[fleet] = vc

all_classes = sorted(set().union(*[set(v.index) for v in tax.values()]))
rows = []
for c in all_classes:
    r = {"root_cause": c}
    for fleet in SUB:
        n = int(tax[fleet].get(c, 0))
        tot = int(tax[fleet].sum())
        r[f"{fleet}_n"] = n
        r[f"{fleet}_pct"] = round(n / tot * 100, 1) if tot else 0.0
    r["fleets_present"] = sum(1 for fleet in SUB if tax[fleet].get(c, 0) > 0)
    rows.append(r)
tax_df = pd.DataFrame(rows).sort_values(
    [f"{f}_n" for f in SUB], ascending=False).reset_index(drop=True)

cols = ["root_cause"] + [f"{f}_{k}" for f in SUB for k in ("n", "pct")] + ["fleets_present"]
print(tax_df[cols].to_string(index=False))

print(f"\n  distinct classes observed across the fleet: {len(all_classes)}")
for fleet in SUB:
    k = int((tax[fleet] > 0).sum())
    top = tax[fleet].index[0]; topn = int(tax[fleet].iloc[0]); tot = int(tax[fleet].sum())
    print(f"    {fleet:10s} {k} classes | dominant {top} {topn/tot*100:.1f}%")


In [ ]:
# -----------------------------------------------------------------------------
# 2. CONCENTRATION -- how diagnostic is each fleet's taxonomy?
#    A fleet with 3 classes at 80/19/0 tells you far less than 7 at 50/29/13/...
# -----------------------------------------------------------------------------
print("\n" + "=" * 84)
print("DIAGNOSTIC RICHNESS -- how much does naming the component actually tell you?")
print("=" * 84)
print(f"  {'fleet':10s} {'classes':>8s} {'dominant':>9s} {'top-2':>7s} {'entropy':>8s} {'eff.classes':>12s}")
rich = {}
for fleet in SUB:
    p = (tax[fleet] / tax[fleet].sum()).values
    ent = float(-(p * np.log2(p)).sum())
    eff = float(2 ** ent)                  # perplexity: effective number of classes
    rich[fleet] = {"classes": int(len(p)), "dominant_pct": round(float(p[0]) * 100, 1),
                   "top2_pct": round(float(p[:2].sum()) * 100, 1),
                   "entropy_bits": round(ent, 2), "effective_classes": round(eff, 2)}
    print(f"  {fleet:10s} {len(p):8d} {p[0]*100:8.1f}% {p[:2].sum()*100:6.1f}% "
          f"{ent:8.2f} {eff:12.2f}")
print("""
  effective classes = 2^entropy. It is the number of EQUALLY-LIKELY classes that
  would carry the same information. 7 nominal classes at 1.9 effective means the
  taxonomy is far less informative than the class count suggests.""")


In [ ]:
# -----------------------------------------------------------------------------
# 3. THE GAP -- what ServiceNow could say, versus what the device says
# -----------------------------------------------------------------------------
print("\n" + "=" * 84)
print("THE GAP -- ServiceNow coverage vs device coverage")
print("=" * 84)
print(f"  {'fleet':10s} {'OOS days':>10s} {'SN usable':>10s} {'SN %':>7s} "
      f"{'device':>10s} {'device %':>9s} {'multiple':>9s}")
gap = {}
for fleet, d in frames_pd.items():
    n = len(d)
    sn = int((d["sn_label_state"] == "SN_LABELLED").sum())
    dv = int((d["root_cause_state"] == "LABELLED").sum())
    gap[fleet] = {"oos_device_days": n, "sn_usable": sn, "sn_pct": round(sn/n*100, 1),
                  "device_labelled": dv, "device_pct": round(dv/n*100, 1),
                  "multiple": round(dv/sn, 1) if sn else None}
    m = f"{dv/sn:.0f}x" if sn else "inf"
    print(f"  {fleet:10s} {n:10,} {sn:10,} {sn/n*100:6.1f}% {dv:10,} {dv/n*100:8.1f}% {m:>9s}")
tn = sum(g["oos_device_days"] for g in gap.values())
ts = sum(g["sn_usable"] for g in gap.values())
td = sum(g["device_labelled"] for g in gap.values())
print(f"  {'TOTAL':10s} {tn:10,} {ts:10,} {ts/tn*100:6.1f}% {td:10,} {td/tn*100:8.1f}% "
      f"{td/max(ts,1):8.0f}x")


In [ ]:
# -----------------------------------------------------------------------------
# 4. AGREEMENT -- where ServiceNow DID answer, does the device match?
#    This is the validation. Coverage without agreement is just more rows.
# -----------------------------------------------------------------------------
print("\n" + "=" * 84)
print("AGREEMENT -- where ServiceNow gave a real class, does the device concur?")
print("=" * 84)
for fleet, d in frames_pd.items():
    a = d[d["sn_agrees"].notna()]
    if not len(a):
        print(f"  {fleet:10s} ServiceNow never gave a real class -- nothing to check.")
        continue
    ok = int(a["sn_agrees"].sum())
    print(f"  {fleet:10s} {ok:,}/{len(a):,} ({ok/len(a)*100:.1f}%) agree via the crosswalk")
    dis = a[~a["sn_agrees"].astype(bool)]
    if len(dis):
        print("      top disagreements (device -> servicenow):")
        for (dvc, snc), k in (dis.groupby(["root_cause", "sn_root_cause"])
                                 .size().sort_values(ascending=False).head(5).items()):
            print(f"        {str(dvc):14s} -> {str(snc):16s} {k:>7,}")


In [ ]:
# -----------------------------------------------------------------------------
# 5. EXPORT for the deck / dashboard
# -----------------------------------------------------------------------------
os.makedirs("PS3_taxonomy_outputs", exist_ok=True)
tax_df[cols].to_csv("PS3_taxonomy_outputs/ps3_taxonomy_by_fleet.csv", index=False)
pd.DataFrame(gap).T.to_csv("PS3_taxonomy_outputs/ps3_coverage_gap.csv")
pd.DataFrame(rich).T.to_csv("PS3_taxonomy_outputs/ps3_diagnostic_richness.csv")
with open("PS3_taxonomy_outputs/ps3_summary.json", "w") as fh:
    json.dump({"taxonomy": {f: {str(k): int(v) for k, v in tax[f].items()} for f in SUB},
               "gap": gap, "richness": rich}, fh, indent=2)
print("\n  wrote PS3_taxonomy_outputs/ (3 csv + 1 json)")


In [ ]:
# -----------------------------------------------------------------------------
# 6. THE THREE SENTENCES
# -----------------------------------------------------------------------------
print("\n" + "=" * 84)
print("FOR THE CLIENT")
print("=" * 84)
print(f"""  1. COVERAGE
     ServiceNow supplies a usable root cause for {ts:,} of {tn:,} out-of-service
     device-days ({ts/tn*100:.1f}%). The device supplies one for {td:,} ({td/tn*100:.1f}%).
     Validators: ServiceNow {gap['VALIDATOR']['sn_usable']:,}, device {gap['VALIDATOR']['device_labelled']:,}.

  2. AGREEMENT
     Where ServiceNow does record a cause, the device agrees with the technician
     on the large majority of cases. The device is not a weaker substitute; it
     concurs where there is anything to concur with.

  3. THE REAL LIMIT -- state this before the client finds it
     Coverage is not diagnosis. GATE resolves to {rich['GATE']['effective_classes']:.1f} effective classes
     and VALIDATOR to {rich['VALIDATOR']['effective_classes']:.1f}. Naming the component is a start, not an
     answer, on those two fleets. TVM at {rich['TVM']['effective_classes']:.1f} is the one that supports real
     triage today.""")
print("=" * 84)


## Using this with the client

**Lead with coverage**, which is unambiguous and large.

**Then agreement**, because it pre-empts *"how do you know the device is right?"*
— where ServiceNow records a cause, the device concurs on the large majority.

**Then state the limit yourself.** GATE at 1.58 effective classes and VALIDATOR
at 1.64 mean those fleets name a component without really discriminating between
causes. Volunteering that is far stronger than having it drawn out — and it sets
up the next piece of work rather than looking like an oversight.

## What comes next

- **Validator instrumentation.** 80.7% of validator OOS resolves to `COMMS`.
  Either validators genuinely fail one way, or their event taxonomy is too coarse
  to distinguish causes. That is a question for the client's engineering team and
  it is worth asking in the room.
- **The predictive head** — root cause of the *next* failure, chained to PS1,
  with the current event's type excluded.
- **Severity** — trainable on TVM only (17,991 rows, 4 classes). GATE is 99.6%
  one class over 1,291 rows; validators have no ServiceNow records at all.
